# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Arxiv has 82 new papers today
          4 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/4 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.17420


extracting tarball to tmp_2611.17420... done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.17420/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.17420/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


Retrieving document from  https://arxiv.org/e-print/2611.17832
extracting tarball to tmp_2611.17832...

 done.


... (3 outputs left out)

/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.17894/blank.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'Funding_Acknowledgements' from 'tmp_2611.17894/Funding_Acknowledgements.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.17894/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


/tmp/ipykernel_2688/2822249172.py:52: LatexWarning: 2611.17894 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.17960/SPE_DR1.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.17960/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


Retrieving document from  https://arxiv.org/e-print/2611.17960
extracting tarball to tmp_2611.17960... done.


/tmp/ipykernel_2688/2822249172.py:52: LatexWarning: 2611.17960 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.17420-b31b1b.svg)](https://arxiv.org/abs/2611.17420) | **Euclid Data Release 1 (DR1). Enhancing deep-learning models: the impact of image processing and training data selection for strong-lens detection**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-16*|
|*Comments*| *30 pages, 13 figures*|
|**Abstract**|            Euclid is expected to discover ~$10^5$ galaxy-galaxy strong gravitational lenses across ~14,000deg$^2$, requiring automated searches with high completeness and low false-positive rates. Despite advances in deep learning, the effects of training-set composition, image representation, and preprocessing on survey-scale detection remain poorly understood. We present an ablation study of the convolutional neural network used in the ECLIPSE strong-lens detection pipeline, evaluating different training configurations on a fixed test set. Using simulated and real Euclid data, we compare VIS-only and pseudo-RGB representations, FITS and JPEG formats, training-set composition, hard-negative selection, and multi-stage training. We also assess ensembling on ~67 million DR1 cutouts. Training-set composition and strategy strongly affect high-Recall performance. Omitting simulation based pre-training reduces high-Recall weighted average Precision by 83.7% for VIS-only and 74.5% for pseudo-RGB models. Negative-sample selection also substantially affects performance, while FITS yields modest gains over JPEG at high Recall (~2-5%). On DR1, the logit-mean ensemble improves average Precision by 28.0% and 39.9% over the individual VIS-only and pseudo-RGB models. Among ~70,000 candidates selected for visual inspection, it recovers 11,256 unique grade A (secure) and B (probable) lenses, compared with 9650 and 10,395 for the individual models. The branches show complementary ranking preferences, particularly for lenses benefiting from near-infrared information. Careful training-set design, realistic negatives, and multi-band ensembling improve survey-scale searches. Our final ensemble achieves ~80% Recall at the operational selection threshold, outperforming ablation models with higher global classification metrics and highlighting the need to optimise performance for realistic survey conditions.[Abridged]         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.17832-b31b1b.svg)](https://arxiv.org/abs/2611.17832) | **Euclid Data Release 1 (DR1): The MERge processing**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-16*|
|*Comments*| *21 pages, 20 Figures*|
|**Abstract**|            The ESA satellite mission \Euclid was launched in July 2023 and started its regular observing mode in February 2024. The data from the \Euclid instruments Visible Camera (VIS) and the Near IR Spectrometer and Photometer (NISP) are complemented by ground-based imaging data in the $u$, $g$ $r$, $i$, and $z$ bands to enable the determination of photometric redshifts and other galaxy properties. This paper describes the MERge processing function (MER PF) that has been developed and run for the Euclid Data Release 1 (DR1). The aim of the MER PF is to detect objects in the \Euclid imaging data, measure their properties, and combine them into a single multi-wavelength catalogue. In this paper we illustrate the design of the MER PF and introduce the different photometric methods applied to the data, which are Kron total flux, aperture photometry on Point Spread Functions (PSF) -matched images, template fitting photometry, and Sérsic fitting photometry. We also present the various morphological measurements and quality flags that improve the scientific content of the output catalogues and better characterize the detected sources. We illustrate the processing and measurements that help to validate the MER PF results, which is an essential step to qualify the output for the public DR1 release. We discuss the global assessment of the photometric homogeneity and provide an internal comparison of our processing results. In addition, we give an extensive overview of the DR1 output data and the various processing scenarios         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.17894-b31b1b.svg)](https://arxiv.org/abs/2611.17894) | **Euclid Data Release 1 (DR1). AstroVink-TREX: Automated strong gravitational lens segmentation as input for mass modelling pipelines**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-16*|
|*Comments*| **|
|**Abstract**|            Strong gravitational lens modelling requires the lens light, lensed source light, and contaminants to be separated in each image before mass modelling codes can fit the system. Within Euclid this separation has so far been done by hand, which does not scale to the more than 15,000 candidates in the Euclid Lens Identification Pipeline and Scientific Exploration (ECLIPSE) catalogue of Data Release 1 (DR1). We present AstroVink-TREX , a neural network that performs this separation automatically and predicts three flux maps, in a standard format directly usable by mass-modelling pipelines. The network uses a hybrid architecture, pairing a vision transformer encoder reused from the lens detection network with a parallel convolutional branch. A custom loss function penalises both flux and structural error, while allowing independent weighting per component. From the network's predictions we also derive the modelling centre, fitting region, conjugate points, and binary maps, providing all the inputs that a modelling pipeline needs to run without manual setup. We evaluate the efficiency of the network on DR1 systems against existing fits and through expert validation. Across approximately 3000 inspected systems, taking 75% agreement between inspectors, just 39 lens models fail because of a segmentation error. Following this validation, the network provides the component separation and modelling setup for every system modelled in DR1. With automated segmentation, lens modelling is no longer limited by manual preprocessing and can now operate at the same scale as lens discovery.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.17960-b31b1b.svg)](https://arxiv.org/abs/2611.17960) | **Euclid Data Release 1 (DR1) : Performance of the redshift measurement and spectroscopic analysis**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-16*|
|*Comments*| *24 pages, 11 figures*|
|**Abstract**|            The European Space Agency's Euclid mission has issued its first major data release(Data Release 1, DR1), providing a transformative view of the extragalactic sky through deep photometry and near-infrared slitless spectroscopy. This paper presents the Euclid DR1 spectroscopic data set and details the evolution of the spectroscopic processing function (SPE PF) pipeline, which is responsible for the automated analysis of 1-dimensional spectra acquired by the Near-Infrared Spectro-Photometer (NISP). The SPE PF pipeline workflow comprises spectroscopic classification, redshift determination, and the systematic measurement of spectral features. Emission line fluxes are derived using two independent methodologies: direct integration (DI) and Gaussian fitting (GF). For sources exhibiting broad H$\alpha$ profiles (velocity dispersion larger than $1500$ km s$^{-1}$), a dedicated `QSO mode' is implemented to robustly deconvolve nuclear and host galaxy components. The data set was validated by cross-matching Euclid results with external spectroscopic catalogues, specifically Dark Energy Spectroscopic Instrument (DESI) DR1, and through an extensive visual inspection campaign of approximately 4000 sources within the Euclid Deep Field North (EDF-N). For high-quality samples selected in the EDF-N, we find a 98.5\% agreement between Euclid and DESI spectroscopic redshifts. This success rate increases to 98.9\% when incorporating pipeline probability and reliability estimators. (abridged)         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_vis_weighted.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_rgb_weighted.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_vis_weighted_zoom_recall_0p75_to_1.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_rgb_weighted_zoom_recall_0p75_to_1.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_vis_weighted.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_rgb_weighted.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_vis_weighted_zoom_recall_0p75_to_1.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_rgb_weighted_zoom_recall_0p75_to_1.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/roc_stage10_vis_weighted.png', 'tmp_2611.17420/./pr_roc_alldr1lenses/roc_stage10_rgb_weighted.png']
copying  tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_vis_weighted.png to _build/html/
copying  tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_rgb_weighted.png to _build/html/
copying  tmp_2611.17420/./pr_ro

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\orcid}[1]{ \orcidlink{#1}}$
$\newcommand{\arraystretch}{1.2}$
$\newcommand{\arraystretch}{1.2}$
$\newcommand{\arraystretch}{1.2}$
$\newcommand{\arraystretch}{1.2}$</div>



<div id="title">

# Euclid Data Release 1 (DR1): Enhancing deep-learning models: the impact of image processing and training data selection for strong-lens detection

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.17420-b31b1b.svg)](https://arxiv.org/abs/2611.17420)<mark>Appeared on: 2026-11-16</mark> -  _30 pages, 13 figures_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** The $\Euclid$ mission is expected to discover ${\sim} 10^5$ galaxy--galaxy strong gravitational lenses across its ${\sim} 14 000$ deg $^2$ survey, requiring efficient automated searches with high completeness and low false-positive rates. Although deep-learning lens finders have achieved remarkable success in recent years, the impact of training-set composition, image representation, and preprocessing on survey-scale lens detection remains poorly understood.We investigate these methodological choices through a systematic ablation study of a convolutional neural network developed for the ECLIPSE strong-lens detection pipeline, systematically varying its training configuration while evaluating the resulting models on a fixed test set. Using simulated and real $\Euclid$ data, we compare VIS-only and pseudo-RGB representations, FITS and JPEG formats, training-set composition, hard-negative selection, and multi-stage training strategies. We further assess model ensembling and evaluate the final classifiers using approximately $67$ million DR1 cutouts.We find that training-set composition and training strategy strongly affect high-Recall performance. Omitting simulation based pre-training reduces high-Recall weighted average Precision by $83.7\%$ for VIS-only and $74.5\%$ for pseudo-RGB models, while negative-sample selection also substantially influences performance. FITS-trained models achieve modest improvements over JPEG-trained counterparts at high Recall ( ${\sim} 2$ -- $5\%$ ). On the full DR1 survey, the logit-mean ensemble improves average Precision by $28.0\%$ and $39.9\%$ relative to the individual VIS-only and pseudo-RGB models, respectively. Among the ${\sim} 70 000$ highest-ranked candidates selected for visual inspection, it recovers $11 256$ unique grade A (secure) and B (probable) lenses, compared with $9650$ and $10 395$ for the individual models, corresponding to $1606$ ( $+ 11.3\%$ ) and $861$ ( $+ 6.0\%$ ) additional systems. The two branches exhibit complementary ranking preferences, particularly for lenses benefiting from near-infrared information.We conclude that careful training-set design, realistic negative examples, and multi-band ensembling improve survey-scale strong-lens searches. Our final ensemble achieves ${\sim} 80\%$ Recall at the operational selection threshold, outperforming ablation models with higher global classification metrics and highlighting the importance of optimising performance for realistic survey conditions.

</div>

<div id="div_fig1">

<img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_vis_weighted.png" alt="Fig9.1" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_rgb_weighted.png" alt="Fig9.2" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_vis_weighted_zoom_recall_0p75_to_1.png" alt="Fig9.3" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_stage10_rgb_weighted_zoom_recall_0p75_to_1.png" alt="Fig9.4" width="25%"/>

**Figure 9. -** *Upper panels*: Stage 1 weighted PR curves for the VIS-only model family (left) and the Pseudo-RGB family (right). Legends report the weighted average Precision. The curves separate most clearly at high Recall, where the practical operating point is determined by how much precision can be retained while still recovering a large fraction of the lenses. *Lower panels*: same as the upper panels but zoomed between $R=0.75$ and $R=1.00$. Legends report the normalised weighted average Precision for $R \in [0.75,1.00]$. The high-$R$ zoom shows that VIS is comparatively robust under several data-set changes, while the Pseudo-RGB branch can degrade sharply if the Stage 1 training set is altered. (*fig:ablation_stage10_pr*)

</div>
<div id="div_fig2">

<img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_vis_weighted.png" alt="Fig7.1" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_rgb_weighted.png" alt="Fig7.2" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_vis_weighted_zoom_recall_0p75_to_1.png" alt="Fig7.3" width="25%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/pr_baseline_rgb_weighted_zoom_recall_0p75_to_1.png" alt="Fig7.4" width="25%"/>

**Figure 7. -** *Upper panels*: baseline Stage 0 weighted PR curves for the VIS-only model family (left) and the Pseudo-RGB family (right). Legends report the weighted average Precision. *Lower panels*: same as the upper panels but zoomed between $R=0.75$ and $R=1.00$. Legends report the normalised weighted average Precision for $R \in [0.75,1.00]$. The high-$R$ zoom shows how some training sets preserve Precision once Recall rises above ${\sim} 0.75$, which is exactly the regime that matters for candidate generation. (*fig:ablation_baseline_pr*)

</div>
<div id="div_fig3">

<img src="tmp_2611.17420/./pr_roc_alldr1lenses/roc_stage10_vis_weighted.png" alt="Fig10.1" width="50%"/><img src="tmp_2611.17420/./pr_roc_alldr1lenses/roc_stage10_rgb_weighted.png" alt="Fig10.2" width="50%"/>

**Figure 10. -** Stage 1 ROC curves for the VIS-only model family (left) and the pseudo-RGB family (right). As in the baseline case, the ROC curves are highly compressed near the upper-left corner, which means that ROC-AUC alone does not separate the ablations cleanly. The low-FPR zoom is the more informative version of this figure because it exposes how much false-positive mass is accumulated before the curve reaches the high-TPR regime. (*fig:ablation_stage10_roc*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.17420"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\linenumbers}[0]\usepackage$</div>



<div id="title">

# Euclid Data Release 1 (DR1): The MERge processing

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.17832-b31b1b.svg)](https://arxiv.org/abs/2611.17832)<mark>Appeared on: 2026-11-16</mark> -  _21 pages, 20 Figures_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** The ESA satellite mission $\Euclid$ was launched in July 2023 and started its regular observing mode in February 2024.The data from the $\Euclid$ instruments Visible Camera (VIS) and the Near IR Spectrometer and Photometer (NISP) are complemented by ground-based imaging data in the $u$ , $g$ $r$ , $i$ , and $z$ bands to enable the determination of photometric redshifts and other galaxy properties. This paper describes the MERge processing function (MER PF) that has been developed and run for the Euclid Data Release 1 (DR1). The aim of the MER PF is to detect objects in the $\Euclid$ imaging data, measure their properties, and combine them into a single multi-wavelength catalogue. In this paper we illustrate the design of the MER PF and introduce the different photometric methods applied to the data, which are Kron total flux, aperture photometry on Point Spread Functions (PSF) -matched images, template fitting photometry, and Sérsic fitting photometry. We also present the various morphological measurements and quality flags that improve the scientific content of the output catalogues and better characterize the detected sources. We illustrate the processing and measurements that help to validate the MER PF results, which is an essential step to qualify the output for the public DR1 release. We discuss the global assessment of the photometric homogeneity and provide an internal comparison of our processing results. In addition, we give an extensive overview of the DR1 output data and the various processing scenarios.

</div>

<div id="div_fig1">

<img src="tmp_2611.17832/./figs/vis_exptime_TILE101837492.png" alt="Fig2" width="100%"/>

**Figure 2. -** The VIS exposure time map for the EDF-N tile 101837492 computed from the layer file. (*figure:vis_expt_TILE101837492*)

</div>
<div id="div_fig2">

<img src="tmp_2611.17832/./figs/VIS_PSF_vs_GAIA_corr_colorbar.png" alt="Fig7" width="100%"/>

**Figure 7. -** Median FWHM of the VIS propagated PSF compared to the median PSF of _ Gaia_ stars measured on the VIS MER mosaics for each tile of EDF-N patch 64. Dashed line represents the one-to-one relation and dotted lines $\pm5\%$ difference. (*fig:VISpsfvalidaion*)

</div>
<div id="div_fig3">

<img src="tmp_2611.17832/./figs/cut_149.920-2.427.png" alt="Fig12.1" width="33%"/><img src="tmp_2611.17832/./figs/cut_149.883-2.458.png" alt="Fig12.2" width="33%"/><img src="tmp_2611.17832/./figs/cut_149.954-2.544.png" alt="Fig12.3" width="33%"/>

**Figure 12. -** Comparison of the DR1 nominal segmentation of the VIS mosaic and the segmentation obtained with a different configuration of the deblending algorithm. The left columns shows the VIS mosaic produced by the MER pipeline, the central column shows the nominal DR1 segmentation, while the right column shows the segmentation map obtained with a different configuration applied to mitigate the mismatch in MER detections between the wide and deep processing. In the central and right columns, sources are color-coded according to their $\texttt{SEGMENTATION\_MAP\_ID}$. Note that in some cases (see bottom panels), the different deblending parametrisation could lead to over-deblended clusters of pixels. (*figure:deblending_wide_vs_deep*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.17832"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

151  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

20  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

2  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
